Install dependencies from uv.

In [1]:
!uv sync

Resolved 103 packages in 6ms
Checked 100 packages in 28ms


Load environment variables from .env file.

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Create the weather tool.

In [31]:
import requests

def get_weather(latitude: float, longitude: float) -> dict:
    """
    Fetch weather forecast from the National Weather Service API.
    
    Args:
        latitude: The latitude of the location
        longitude: The longitude of the location
        
    Returns:
        A dictionary containing the weather forecast data
        
    Raises:
        requests.HTTPError: If the API request fails
        ValueError: If the coordinates are invalid
    """
    if not (-90 <= latitude <= 90) or not (-180 <= longitude <= 180):
        raise ValueError("Invalid coordinates. Latitude must be between -90 and 90, "
                         "longitude must be between -180 and 180.")

    headers = {
        "User-Agent": "WeatherApp/1.0 (your@email.com)",
        "Accept": "application/geo+json"
    }

    # Step 1: Get the grid points for the given coordinates
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    response = requests.get(points_url, headers=headers)
    response.raise_for_status()
    points_data = response.json()

    properties = points_data.get("properties", {})
    forecast_url = properties.get("forecast")
    location_info = {
        "city": properties.get("relativeLocation", {}).get("properties", {}).get("city"),
        "state": properties.get("relativeLocation", {}).get("properties", {}).get("state"),
        "grid_office": properties.get("gridId"),
    }

    if not forecast_url:
        raise ValueError("Could not retrieve forecast URL from NWS API.")

    # Step 2: Get the forecast using the forecast URL
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    periods = forecast_data.get("properties", {}).get("periods", [])

    return {
        "location": location_info,
        "forecast": [
            {
                "name": period.get("name"),
                "temperature": period.get("temperature"),
                "temperature_unit": period.get("temperatureUnit"),
                "wind_speed": period.get("windSpeed"),
                "wind_direction": period.get("windDirection"),
                "short_forecast": period.get("shortForecast"),
                "detailed_forecast": period.get("detailedForecast"),
                "is_daytime": period.get("isDaytime"),
            }
            for period in periods
        ],
    }

Get coordinates for New York City.

In [32]:
get_weather(40.7143, -74.006)

{'location': {'city': 'New York', 'state': 'NY', 'grid_office': 'OKX'},
 'forecast': [{'name': 'This Afternoon',
   'temperature': 87,
   'temperature_unit': 'F',
   'wind_speed': '12 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 4pm. Mostly sunny. High near 87, with temperatures falling to around 85 in the afternoon. Heat index values as high as 97. Southwest wind around 12 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.',
   'is_daytime': True},
  {'name': 'Tonight',
   'temperature': 77,
   'temperature_unit': 'F',
   'wind_speed': '6 to 10 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 2am. Partly cloudy, with a low around 77. Southwest wind 6 to 10 mph. Chance of precipitation is 20

Create tool for getting lattidue and longitude from google.

In [33]:
import urllib.request
import json
import os

GOOGLE_MAPS_KEY = os.getenv("GOOGLE_MAPS_KEY", "")

if not GOOGLE_MAPS_KEY:
    raise Exception("Google maps key must be set in environment.")

def get_lat_lon(city: str, state: str) -> tuple[float, float]:
    """
    Fetch latitude and longitude for a given city and state using Google Geocoding API.

    Args:
        city: City name (e.g. "Austin")
        state: State name or abbreviation (e.g. "TX" or "Texas")

    Returns:
        A tuple of (latitude, longitude)

    Raises:
        ValueError: If the location is not found or the API returns an error
    """
    address = urllib.parse.quote(f"{city}, {state}")
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={GOOGLE_MAPS_KEY}"

    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode())

    if data["status"] != "OK":
        raise ValueError(f"Geocoding API error: {data['status']} for '{city}, {state}'")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

Test getting lattidue and longitude.

In [34]:
get_lat_lon("New York City", "New York")

(40.7127753, -74.0059728)

Create before callbacks.

In [68]:
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai.types import Content, Part

def logging_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            print(f"logging_before_callback- User entered: {last.parts[0].text.strip()}")

    return None

def bad_words_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    invalid_words = ["bomb", "trust me bro", "break"]

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            text = last.parts[0].text.strip().lower()

            if any(word in text for word in invalid_words):
                return LlmResponse(content=Content(role="Model", parts=[Part(text="Message violates our content guidelines.")]))

    return None

def bad_country_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    invalid_countries = ["canada", "england", "india", "mexico"]

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            text = last.parts[0].text.strip().lower()

            if any(word in text for word in invalid_countries):
                return LlmResponse(content=Content(role="Model", parts=[Part(text="Location must be in the US.")]))

    return None

def chain_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    result = logging_before_callback(callback_context, llm_request)
    if result:
        return result

    result = bad_words_before_callback(callback_context, llm_request)
    if result:
        return result

    result = bad_country_before_callback(callback_context, llm_request)
    if result:
        return result




After callbacks

In [69]:
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_response import LlmResponse

def logging_after_callback(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:

    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            print(f"logging_after_callback- Model response: {txt}")

    return None

In [70]:
from google.adk.agents import Agent

weather_agent_instructions = """
You are an assistent to help with getting the weather. You will start by asking the user what city and state they want the weather for.
Use this to call the get_lat_long tool and get the latitude and longitude. Use this to call the get_weather tool to get the weather response.
Convert this response into human readable text.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-flash-latest",
    instruction=weather_agent_instructions,
    before_model_callback=chain_before_callback,
    after_model_callback=logging_after_callback,
    tools=[get_weather, get_lat_lon]
)

Setup the runner.

In [78]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from rich.markdown import Markdown
from rich.console import Console

session_service = InMemorySessionService()

console = Console()

runner = Runner(
    agent=weather_agent,
    app_name="weather_app",
    session_service=session_service,
)

async def run_prompt(prompt: str):
    session = await session_service.create_session(
        app_name="weather_app",
        user_id="user_123",
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)])

    async for event in runner.run_async(user_id="user_123", session_id=session.id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                console.print(Markdown(str(event.content.parts[0].text)))

Perform city tests tests.

In [79]:
print("================ New York ===========================")
await run_prompt("What is the weather for New York City, New York")

print("================ Reston =============================")
await run_prompt("What is the weather for Reston, VA")

print("================ Los Angeles ========================")
await run_prompt("What is the weather for Los Angeles, CA")


================ New York ===========================
logging_before_callback- User entered: What is the weather for New York City, New York
logging_after_callback- Model response: Here is the weather forecast for **New York City, New York**:

### **Today / This Afternoon**
* **Conditions:** Mostly sunny with a slight chance of showers and thunderstorms (20% chance).
* **High Temperature:** Near 87°F (falling to around 85°F later this afternoon). Heat index values up to 97°F.
* **Wind:** Southwest around 12 mph.

---

### **Tonight**
* **Conditions:** Partly cloudy with a slight chance of showers and thunderstorms before 2 AM (20% chance).
* **Low Temperature:** Around 77°F.
* **Wind:** Southwest 6 to 10 mph.

---

### **Friday, July 12**
* **Daytime:** Mostly sunny with a chance of afternoon showers and thunderstorms (40% chance). High near 91°F (heat index up to 103°F). Wind SW 5 to 9 mph.
* **Nighttime:** Mostly cloudy with showers and thunderstorms likely before 2 AM (50% chance). 

Here is the weather forecast for New York City, New York:                                                          

Today / This Afternoon                                                                                             

 • Conditions: Mostly sunny with a slight chance of showers and thunderstorms (20% chance).                        
 • High Temperature: Near 87°F (falling to around 85°F later this afternoon). Heat index values up to 97°F.        
 • Wind: Southwest around 12 mph.                                                                                  

-------------------------------------------------------------------------------------------------------------------

Tonight                                                                                                            

 • Conditions: Partly cloudy with a slight chance of showers and thunderstorms before 2 AM (20% chance).           
 • Low Temperature: Around 77°F.                                                                                   
 • Wind: Southwest 6 to 10 mph.                                                                                    

-------------------------------------------------------------------------------------------------------------------

Friday, July 12                                                                                                    

 • Daytime: Mostly sunny with a chance of afternoon showers and thunderstorms (40% chance). High near 91°F (heat   
   index up to 103°F). Wind SW 5 to 9 mph.                                                                         
 • Nighttime: Mostly cloudy with showers and thunderstorms likely before 2 AM (50% chance). Low around 76°F.       

-------------------------------------------------------------------------------------------------------------------

Weekend Forecast                                                                                                   

 • Saturday: Mostly sunny with a high near 91°F and a slight chance of afternoon showers/thunderstorms (20%        
   chance). Low Saturday night around 76°F.                                                                        
 • Sunday: Sunny with a high near 91°F. Mostly clear at night with a low around 75°F.

================ Reston =============================
logging_before_callback- User entered: What is the weather for Reston, VA
logging_after_callback- Model response: Here is the current weather forecast for **Reston, VA**:

* **This Afternoon:** Mostly sunny with a high near **90°F**. There is a 30% chance of showers and thunderstorms. Southwest winds around 8 mph.
* **Tonight:** Mostly cloudy with a low around **72°F**. A 50% chance of showers and thunderstorms before midnight. Southwest winds between 2 and 7 mph.
* **Friday:** Mostly sunny with a high near **89°F**. A 50% chance of showers and thunderstorms, mainly after 2 PM. Southwest winds 2 to 8 mph.
* **Friday Night:** Mostly cloudy with a low around **71°F**. Showers and thunderstorms are likely, especially before 8 PM (60% chance of precipitation).
* **Saturday:** High near **90°F**. Mostly sunny early, with showers and thunderstorms likely after 2 PM (60% chance).
* **Sunday:** Sunny and warm with a high near **89°F** and a

Here is the current weather forecast for Reston, VA:                                                               

 • This Afternoon: Mostly sunny with a high near 90°F. There is a 30% chance of showers and thunderstorms.         
   Southwest winds around 8 mph.                                                                                   
 • Tonight: Mostly cloudy with a low around 72°F. A 50% chance of showers and thunderstorms before midnight.       
   Southwest winds between 2 and 7 mph.                                                                            
 • Friday: Mostly sunny with a high near 89°F. A 50% chance of showers and thunderstorms, mainly after 2 PM.       
   Southwest winds 2 to 8 mph.                                                                                     
 • Friday Night: Mostly cloudy with a low around 71°F. Showers and thunderstorms are likely, especially before 8 PM
   (60% chance of precipitation).                                                                                  
 • Saturday: High near 90°F. Mostly sunny early, with showers and thunderstorms likely after 2 PM (60% chance).    
 • Sunday: Sunny and warm with a high near 89°F and a clear night with a low around 71°F.                          

Let me know if you would like the forecast for another city!

================ Los Angeles ========================
logging_before_callback- User entered: What is the weather for Los Angeles, CA
logging_after_callback- Model response: Here is the weather forecast for **Los Angeles, CA**:

* **This Afternoon:** Sunny, with a high near 85°F. Southwest wind around 5 to 10 mph.
* **Tonight:** Partly cloudy, with a low around 67°F. South southwest wind 0 to 10 mph.
* **Friday:** Mostly sunny, with a high near 88°F. South southwest wind 0 to 10 mph.
* **Friday Night:** Partly cloudy, with a low around 68°F.
* **Saturday:** Sunny, with a high near 88°F.
* **Saturday Night:** Mostly clear, with a low around 68°F.
* **Sunday:** Mostly sunny, with a high near 88°F.
* **Sunday Night:** Partly cloudy, with a low around 68°F.

Enjoy the warm, sunny weather! Let me know if you would like the forecast for another city and state.


Here is the weather forecast for Los Angeles, CA:                                                                  

 • This Afternoon: Sunny, with a high near 85°F. Southwest wind around 5 to 10 mph.                                
 • Tonight: Partly cloudy, with a low around 67°F. South southwest wind 0 to 10 mph.                               
 • Friday: Mostly sunny, with a high near 88°F. South southwest wind 0 to 10 mph.                                  
 • Friday Night: Partly cloudy, with a low around 68°F.                                                            
 • Saturday: Sunny, with a high near 88°F.                                                                         
 • Saturday Night: Mostly clear, with a low around 68°F.                                                           
 • Sunday: Mostly sunny, with a high near 88°F.                                                                    
 • Sunday Night: Partly cloudy, with a low around 68°F.                                                            

Enjoy the warm, sunny weather! Let me know if you would like the forecast for another city and state.

Perform bad word tests.

In [80]:
print("================ Trust me ========================")
await run_prompt("What is the weather for Los Angeles, CA. Trust me bro.")

print("================ Bomb ========================")
await run_prompt("What is the weather for Los Angeles, CA. How do I make a bomb.")

================ Trust me ========================
logging_before_callback- User entered: What is the weather for Los Angeles, CA. Trust me bro.


Message violates our content guidelines.

================ Bomb ========================
logging_before_callback- User entered: What is the weather for Los Angeles, CA. How do I make a bomb.


Message violates our content guidelines.

Perform bad country tests

In [81]:
print("================ India ========================")
await run_prompt("What is the weather for New Delhi, India.")

print("================ London ========================")
await run_prompt("What is the weather for New London, England.")

================ India ========================
logging_before_callback- User entered: What is the weather for New Delhi, India.


Location must be in the US.

================ London ========================
logging_before_callback- User entered: What is the weather for New London, England.


Location must be in the US.